In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 3, Finished, Available, Finished, False)

In [3]:
silver_matches = spark.read.table("silver_matches")
silver_deliveries = spark.read.table("silver_deliveries")

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 5, Finished, Available, Finished, False)

In [4]:
display(silver_matches)
display(silver_deliveries)

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bf920055-8a5f-4342-a4ec-bb24191aff0a)

SynapseWidget(Synapse.DataFrame, 015131b2-ec5d-4999-866f-3da211ef04b7)

# 

# **Batsman Career Statistics**

| Metric      | Logic                  |
| ----------- | ---------------------- |
| Runs        | SUM(batsman_runs)      |
| Balls Faced | Count legal deliveries |
| Strike Rate | (runs/balls)*100       |
| Fours       | batsman_runs == 4      |
| Sixes       | batsman_runs == 6      |
| 50s         | optional later         |
| 100s        | optional later         |

In [10]:
from pyspark.sql.functions import col, sum, when, count, round

# Step 1: Aggregate data at the Match-level per Batsman
batsman_match_stats = silver_deliveries.groupBy("matchId", "batsman").agg(
    sum("batsman_runs").alias("match_runs"),
    count(when(col("isWide") != 1, True)).alias("match_balls_faced"),
    sum(when(col("batsman_runs") == 4, 1).otherwise(0)).alias("match_fours"),
    sum(when(col("batsman_runs") == 6, 1).otherwise(0)).alias("match_sixes")
)

# Step 2: Aggregate Match-level data into Career-level Statistics
batsman_stats = batsman_match_stats.groupBy("batsman").agg(
    sum("match_runs").alias("total_runs"),
    sum("match_balls_faced").alias("balls_faced"),
    sum("match_fours").alias("fours"),
    sum("match_sixes").alias("sixes"),
    sum(when((col("match_runs") >= 50) & (col("match_runs") < 100), 1).otherwise(0)).alias("fifty_plus_scores"),
    sum(when(col("match_runs") >= 100, 1).otherwise(0)).alias("hundreds")
)

# Step 3: Calculate the overall Career Strike Rate
batsman_stats = batsman_stats.withColumn(
    "strike_rate", 
    round((col("total_runs") / col("balls_faced")) * 100, 2)
)

# Display the finalized dataframe
display(batsman_stats)

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e519687e-f3a0-4f6b-947e-a150091890b6)

In [11]:
batsman_stats.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_batsman_stats")

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 13, Finished, Available, Finished, False)

# **Bowler Career Statistics**

| Metric        | Logic                   |
| ------------- | ----------------------- |
| Wickets       | count(is_wicket=1)      |
| Runs Conceded | sum(total_runs)         |
| Balls Bowled  | legal deliveries        |
| Overs         | balls/6                 |
| Economy       | runs/overs              |
| Dot Ball %    | dot balls / total balls |


In [14]:
bowler_stats = silver_deliveries.groupBy("bowler").agg(
    sum(
        when(~col("dismissal_kind").isin(["None", "run out", "retired hurt", "retired out"]), 1).otherwise(0)
    ).alias("wickets"),
    
    sum(
        col("batsman_runs") + 
        when(col("isWide") == 1, col("extras")).otherwise(0) + 
        when(col("isNoBall") == 1, col("extras")).otherwise(0)
    ).alias("runs_conceded"),
    
    count(
        when((col("isWide") != 1) & (col("isNoBall") != 1), True)
    ).alias("balls_bowled"),
    
    sum(
        when((col("batsman_runs") == 0) & (col("isWide") != 1) & (col("isNoBall") != 1), 1).otherwise(0)
    ).alias("dot_balls")
)

display(bowler_stats)

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 60014e1a-a518-408d-b03d-3db0a311a3e6)

In [17]:
bowler_stats = bowler_stats \
    .withColumn(
        "overs",
        round(col("balls_bowled") / 6, 2)
    ) \
    .withColumn(
        "economy",
        round(col("runs_conceded") / (col("balls_bowled") / 6), 2)
    ) \
    .withColumn(
        "dot_ball_percentage",
        round((col("dot_balls") / col("balls_bowled")) * 100, 2)
    )

display(bowler_stats)

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bac85d96-ddd4-4d46-bdf8-db799b636510)

In [18]:
bowler_stats.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_bowler_stats")

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 20, Finished, Available, Finished, False)

# **Team Season Performance**

In [19]:
team_wins = silver_matches.groupBy("season", "winner").agg(
    count("*").alias("wins")
)

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 21, Finished, Available, Finished, False)

In [21]:
team_wins = team_wins.withColumnRenamed("winner", "team")
display(team_wins)

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ae8b0a95-0b88-498c-b3fd-1f0ffc16980e)

In [24]:
team1_matches = silver_matches.select(
    col("season"),
    col("team1").alias("team")
)
team2_matches = silver_matches.select(
    col("season"),
    col("team2").alias("team")
)
all_team_matches = team1_matches.union(team2_matches)

matches_played = all_team_matches.groupBy(
    "season",
    "team"
).agg(
    count("*").alias("matches_played")
)

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 26, Finished, Available, Finished, False)

In [25]:
team_performance = matches_played.join(
    team_wins,
    ["season", "team"],
    "left"
).fillna(0)

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 27, Finished, Available, Finished, False)

In [26]:
team_performance = team_performance.withColumn(
    "win_percentage",
    round(
        (col("wins") / col("matches_played")) * 100,
        2
    )
)

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 28, Finished, Available, Finished, False)

In [27]:
team_performance.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("season") \
    .saveAsTable("gold_team_season_performance")

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 29, Finished, Available, Finished, False)

In [28]:
%%sql
OPTIMIZE gold_batsman_stats;
OPTIMIZE gold_bowler_stats;
OPTIMIZE gold_team_season_performance;

StatementMeta(, f4009536-2cdc-459e-937b-380bec94d49f, 32, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>